In [1]:
# Note: 
# 1. You need to be connected to the HKU network or HKU VPN to access the chenlin04 server.
# 2. Do not upload chenlin04_student_login.txt to any public repository, such as GitHub and LLMs.
# 3. Contact Boao (boaozhan@connect.hku.hk) for any questions related to accessing/using the LinkUp data.

In [1]:
import pandas as pd
import numpy as np
import requests
import clickhouse_connect
from collections import namedtuple

In [2]:
class SimpleResult:
    def __init__(self, json_data):
        self.result_rows = json_data.get('data', [])
        self.column_names = [meta['name'] for meta in json_data.get('meta', [])]

class SimpleClient:
    def __init__(self, host, username, password, **kwargs):
        self.base_url = f"http://{host}:8123/"
        self.auth = {
            'user': username,
            'password': password
        }
        
    def query(self, query_string):
        params = self.auth.copy()
        # Use JSON format to get metadata (column names) easily
        params['query'] = query_string + " FORMAT JSON"
        
        response = requests.get(self.base_url, params=params)
        
        if response.status_code != 200:
            raise Exception(f"Query failed: {response.text}")
            
        return SimpleResult(response.json())

# Initialize the client
with open('chenlin04_student_login.txt', 'r') as f:
    conn_info = eval(f.read())
    try:
        # Try to connect with clickhouse first
        client = clickhouse_connect.get_client(**conn_info)
        print("Connected using standard clickhouse_connect")
    except Exception as e:
        # If it doesn't work out, use requests
        print(f"Standard connection failed: {e}")
        print("Falling back to SimpleClient (lightweight mode)...")
        client = SimpleClient(**conn_info)
        print("Connected using SimpleClient")

Connected using standard clickhouse_connect


In [3]:
res = client.query("""show tables from fina4359_linkup_202603""")
tables = pd.DataFrame(res.result_rows, columns=res.column_names)
tables

,name
0,company_id_compustat_identifiers
1,compustat_na_quarterly_all_firms_since_2000
2,country_state_code_mapping
3,crsp_daily_ret_all_firms_since_2000
4,crsp_monthly_ret_all_firms_since_2000
...,...
378,firm_70345
379,firm_70437
380,firm_7126
381,firm_8417


In [4]:
query = """
WITH cid AS
    (SELECT 
        toInt64(REPLACE(name, 'firm_', '')) as firm_id
    FROM system.tables 
    WHERE database = 'fina4359_linkup_202603' 
    AND name LIKE 'firm_%')
SELECT company_id, toString(floor(naics / 10000)) as sector
FROM company_id_compustat_identifiers 
WHERE company_id IN (SELECT firm_id FROM cid)
"""

res = client.query(query)
firms = pd.DataFrame(res.result_rows, columns=res.column_names).dropna()
firms

,company_id,sector
0,20238,52
1,43730,32
2,44223,33
3,12903,54
4,17378,33
...,...,...
377,27814,33
378,26489,33
379,23091,52
380,24312,22


In [25]:
len(firms['sector'].unique())

24

In [5]:
def check_table_schema(company_id):
    """
    Check the actual data types and sample values
    """
    
    # Check column types
    query = f"""
    DESCRIBE TABLE firm_{company_id}
    """
    
    result = client.query(query)
    print("Table schema:")
    for row in result.result_rows:
        print(f"  {row[0]}: {row[1]}")
    
    # Check sample data
    query = f"""
    SELECT 
        created,
        last_seen,
        toTypeName(created) as created_type,
        toTypeName(last_seen) as last_seen_type
    FROM firm_{company_id}
    LIMIT 5
    """
    
    result = client.query(query)
    print("\nSample data:")
    for row in result.result_rows:
        print(f"  created: {row[0]} ({row[2]}), last_seen: {row[1]} ({row[3]})")
    
    # Check date range
    query = f"""
    SELECT 
        MIN(toDate(created)) as min_date,
        MAX(toDate(created)) as max_date,
        COUNT(*) as total_jobs
    FROM firm_{company_id}
    WHERE created IS NOT NULL
    """
    
    result = client.query(query)
    row = result.result_rows[0]
    print(f"\nDate range: {row[0]} to {row[1]}")
    print(f"Total jobs: {row[2]}")

# Run diagnostic
check_table_schema(70345)

Table schema:
  hash: Nullable(String)
  onet: Int64
  country: Int64
  state: Int64
  zip: Int64
  created: Nullable(String)
  last_updated: Nullable(String)
  updates: Int64
  last_seen: Nullable(String)
  missing: Nullable(String)
  description: Nullable(String)

Sample data:
  created: 2021-08-11 (Nullable(String)), last_seen: 2021-10-20 (Nullable(String))
  created: 2021-08-11 (Nullable(String)), last_seen: 2021-11-01 (Nullable(String))
  created: 2021-08-11 (Nullable(String)), last_seen: 2021-11-07 (Nullable(String))
  created: 2021-08-11 (Nullable(String)), last_seen: 2021-09-28 (Nullable(String))
  created: 2021-11-03 (Nullable(String)), last_seen: 2022-01-13 (Nullable(String))

Date range: 2021-08-11 to 2022-09-24
Total jobs: 15


In [6]:
def aggregate_monthly_jobs_openings(company_id, start_date='2010-01-01', end_date='2022-12-31'):
    query = f"""
    WITH 
        jobs AS (
            SELECT 
                hash,
                toDate(created) as created_date,
                toDate(last_seen) as last_seen_date
            FROM firm_{company_id}
            WHERE created IS NOT NULL 
                AND created != ''
                AND toDate(created) IS NOT NULL
        ),
        -- Generate month indices using arrayJoin
        month_indices AS (
            SELECT arrayJoin(range(0, dateDiff('month', toDate('{start_date}'), toDate('{end_date}')) + 1)) as idx
        ),
        months AS (
            SELECT 
                toDate(addMonths(toDate('{start_date}'), idx)) as month_start,
                toLastDayOfMonth(toDate(addMonths(toDate('{start_date}'), idx))) as month_end
            FROM month_indices
        )
    SELECT 
        m.month_start as month,
        COUNT(DISTINCT j.hash) as active_jobs,
        COUNT(DISTINCT CASE 
            WHEN j.created_date BETWEEN m.month_start AND m.month_end 
            THEN j.hash 
        END) as new_jobs,
        COUNT(DISTINCT CASE 
            WHEN j.last_seen_date BETWEEN m.month_start AND m.month_end 
            THEN j.hash 
        END) as closed_jobs,
        COUNT(DISTINCT CASE 
            WHEN j.created_date <= m.month_start 
                AND (j.last_seen_date IS NULL OR j.last_seen_date >= m.month_end)
            THEN j.hash 
        END) as jobs_open_full_month
    FROM months m
    CROSS JOIN jobs j
    WHERE j.created_date <= m.month_end
        AND COALESCE(j.last_seen_date, toDate('{end_date}') + 1) >= m.month_start
    GROUP BY m.month_start
    ORDER BY m.month_start
    """
    
    result = client.query(query)
    df = pd.DataFrame(
        result.result_rows, 
        columns=['month', 'active_jobs', 'new_jobs', 'closed_jobs', 'jobs_open_full_month']
    )
    return df

In [7]:
def aggregate_multiple_firms_monthly(company_ids, start_date='2010-01-01', end_date='2022-12-31'):
    """
    Aggregate monthly job openings for multiple firms simultaneously
    """
    
    # Create UNION ALL query for multiple firms
    union_queries = []
    for company_id in company_ids:
        union_queries.append(f"""
            SELECT
                {company_id} as company_id,
                hash,
                toDate(created) as created_date,
                toDate(last_seen) as last_seen_date
            FROM firm_{company_id}
            WHERE created IS NOT NULL 
                AND created != ''
                AND toDate(created) IS NOT NULL
                AND toDate(created) <= toDate('{end_date}')
        """)
    
    union_query = " UNION ALL ".join(union_queries)
    
    query = f"""
    WITH all_jobs AS (
        {union_query}
    ),
        month_indices AS (
                SELECT arrayJoin(range(0, dateDiff('month', toDate('{start_date}'), toDate('{end_date}')) + 1)) as idx
            ),
        months AS (
            SELECT 
                toDate(addMonths(toDate('{start_date}'), idx)) as month_start,
                toLastDayOfMonth(toDate(addMonths(toDate('{start_date}'), idx))) as month_end
            FROM month_indices
        )
    SELECT 
        m.month_start as month,
        COUNT(DISTINCT j.hash) as active_jobs
    FROM months m, all_jobs j
    WHERE j.created_date <= m.month_end
        AND (j.last_seen_date IS NULL OR j.last_seen_date >= m.month_start)
    GROUP BY m.month_start
    ORDER BY m.month_start
    """
    
    res = client.query(query)
    df = pd.DataFrame(res.result_rows, columns=res.column_names)
    
    return df

# Example usage
company_list = [8417, 70345, 70437]  # Replace with actual company IDs
multi_firm_df = aggregate_multiple_firms_monthly(company_list)
multi_firm_df

,month,active_jobs
0,2015-04-01,13
1,2015-05-01,23
2,2015-06-01,29
3,2015-07-01,32
4,2015-08-01,30
...,...,...
63,2022-06-01,198
64,2022-07-01,435
65,2022-08-01,258
66,2022-09-01,324


In [8]:
def get_sector_monthly_jobs(start_date='2010-01-01', end_date='2022-12-31'):
    """
    Two-step approach: first get firm-sector mapping, then aggregate
    """
    
    # Step 1: Get firm to sector mapping
    mapping_query = """
    SELECT 
        toInt64(REPLACE(t.name, 'firm_', '')) as company_id,
        toString(floor(c.naics / 10000)) as sector
    FROM system.tables t
    JOIN company_id_compustat_identifiers c
        ON toInt64(REPLACE(t.name, 'firm_', '')) = c.company_id
    WHERE t.database = 'fina4359_linkup_202603' 
        AND t.name LIKE 'firm_%'
        AND c.naics IS NOT NULL
    """
    
    mapping_result = client.query(mapping_query)
    firm_sector = {row[0]: row[1] for row in mapping_result.result_rows if row[1] != np.nan}
    
    # Step 2: Aggregate for each firm and combine
    all_data = []
    
    for company_id, sector in firm_sector.items():
        # print(f"Processing firm {company_id} (sector {sector})...")
        
        query = f"""
        WITH 
            months AS (
                SELECT 
                    toDate(addMonths(toDate('{start_date}'), idx)) as month_start,
                    toLastDayOfMonth(toDate(addMonths(toDate('{start_date}'), idx))) as month_end
                FROM (
                    SELECT arrayJoin(range(0, dateDiff('month', toDate('{start_date}'), toDate('{end_date}')) + 1)) as idx
                )
            ),
            jobs AS (
                SELECT 
                    hash,
                    toDate(created) as created_date,
                    toDate(last_seen) as last_seen_date
                FROM firm_{company_id}
                WHERE created IS NOT NULL 
                    AND created != ''
                    AND toDate(created) IS NOT NULL
            )
        SELECT 
            m.month_start as month,
            COUNT(DISTINCT j.hash) as active_jobs
        FROM months m
        CROSS JOIN jobs j
        WHERE j.created_date <= m.month_end
            AND (j.last_seen_date IS NULL OR j.last_seen_date >= m.month_start)
        GROUP BY m.month_start
        ORDER BY m.month_start
        """
        
        result = client.query(query)
        df = pd.DataFrame(result.result_rows, columns=['month', 'active_jobs'])
        df['sector'] = sector
        df['company_id'] = company_id
        all_data.append(df)
    
    # Combine all data
    final_df = pd.concat(all_data, ignore_index=True)
    
    # Aggregate by sector and month
    sector_monthly = final_df.groupby(['month', 'sector'])['active_jobs'].sum().reset_index()
    
    return sector_monthly

In [ ]:
def calculate_rs_ratio(df, lookback=5):
    """
    Calculate Job Relative Strength Ratio
    df: DataFrame with columns: date, sector, active_jobs
    lookback: rolling window in months
    """
    
    # Pivot to get sectors as columns
    pivot_df = df.pivot(index='month', columns='sector', values='active_jobs').fillna(0)
    
    # Calculate monthly percentage change
    pct_change = pivot_df.pct_change()
    
    # Calculate market average (equal-weighted or posting-weighted)
    market_avg_pct_change = pct_change.mean(axis=1)
    
    # Calculate J-RS-Ratio for each sector
    rs_ratio = pd.DataFrame(index=pivot_df.index)
    
    for sector in pivot_df.columns:
        # RS-Ratio = (Sector % Change / Market % Change) * 100, then smoothed
        raw_ratio = (pct_change[sector] / market_avg_pct_change) * 100
        
        # Apply J-D metric (similar to KDJ indicator) or simple moving average
        rs_ratio[sector] = raw_ratio.rolling(window=lookback, min_periods=5).mean()
        
        # Normalize to center around 100 (like Bloomberg)
        rs_ratio[sector] = 100 + (rs_ratio[sector] - rs_ratio[sector].rolling(window=3).mean()) / rs_ratio[sector].rolling(window=3).std()
    
    return rs_ratio

In [10]:
def calculate_rs_momentum(rs_ratio):
    """
    Calculate Job Relative Strength Momentum
    Measures the rate of change of RS-Ratio
    """
    
    rs_momentum = pd.DataFrame(index=rs_ratio.index)
    
    for sector in rs_ratio.columns:
        rs_momentum[sector] = (rs_ratio[sector] / rs_ratio[sector].shift(4)) * 100
        
    return rs_momentum

In [11]:
def calculate_enhanced_features(df):
    """
    Add additional job posting metrics for more robust signals
    """
    
    pivot_df = df.pivot(index='job_date', columns='sector', values='sector_total_postings').fillna(0)
    
    features = {}
    
    # 1. Job Posting Breadth
    features['breadth'] = (pivot_df > pivot_df.rolling(window=20, min_periods=5).mean()).sum(axis=1) / len(pivot_df.columns)
    
    # 2. High-Skill Job Ratio (using ONET codes)
    # This would require joining with ONET classification
    # high_skill_ratio = ...
    
    # 3. Job Duration Trend (using last_seen - created)
    # avg_duration = ...
    
    # 4. New vs. Repeat Postings
    # new_posting_ratio = ...
    
    return features

In [12]:
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta

def create_job_rrg(rs_ratio, rs_momentum, date, tail_days=30):
    """
    Create a Relative Rotation Graph for job postings
    """
    
    # Get data for the specific date
    current_rs = rs_ratio.loc[date]
    current_momentum = rs_momentum.loc[date]
    
    # Get historical path for tail (last tail_days)
    start_date = date - timedelta(days=tail_days)
    historical_rs = rs_ratio.loc[start_date:date]
    historical_momentum = rs_momentum.loc[start_date:date]
    
    # Create figure
    fig = go.Figure()
    
    # Add quadrant lines
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    fig.add_vline(x=100, line_dash="dash", line_color="gray", opacity=0.5)
    
    # Add quadrant labels
    fig.add_annotation(x=105, y=0.5, text="Leading", showarrow=False, font=dict(size=12, color="green"))
    fig.add_annotation(x=95, y=0.5, text="Weakening", showarrow=False, font=dict(size=12, color="orange"))
    fig.add_annotation(x=95, y=-0.5, text="Lagging", showarrow=False, font=dict(size=12, color="red"))
    fig.add_annotation(x=105, y=-0.5, text="Improving", showarrow=False, font=dict(size=12, color="blue"))
    
    # Add tail paths for each sector
    for sector in rs_ratio.columns:
        tail_x = historical_rs[sector]
        tail_y = historical_momentum[sector]
        
        # Add tail line
        fig.add_trace(go.Scatter(
            x=tail_x,
            y=tail_y,
            mode='lines',
            line=dict(width=1, color='lightgray'),
            showlegend=False,
            hoverinfo='none'
        ))
        
        # Add current position marker
        fig.add_trace(go.Scatter(
            x=[current_rs[sector]],
            y=[current_momentum[sector]],
            mode='markers+text',
            marker=dict(size=12, 
                    #    color=get_sector_color(sector),
                       line=dict(width=2, color='black')),
            text=sector[:4],  # Abbreviated sector name
            textposition="top center",
            name=sector,
            hovertemplate=f'<b>{sector}</b><br>' +
                          f'J-RS-Ratio: %{{x:.1f}}<br>' +
                          f'J-RS-Momentum: %{{y:.2f}}<br>' +
                          '<extra></extra>'
        ))
    
    # Update layout
    fig.update_layout(
        title=f'Job Posting Relative Rotation Graph (RRG) - {date.strftime("%Y-%m-%d")}',
        xaxis_title='J-RS-Ratio (Relative Strength)',
        yaxis_title='J-RS-Momentum',
        width=900,
        height=800,
        hovermode='closest',
        showlegend=True,
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.05
        )
    )
    
    # Set axis ranges with some padding
    fig.update_xaxes(range=[70, 130])
    fig.update_yaxes(range=[-2, 2])
    
    return fig

def get_sector_color(sector):
    """Assign consistent colors to sectors"""
    colors = {
        'Technology': 'blue',
        'Finance': 'green', 
        'Healthcare': 'red',
        'Energy': 'orange',
        'Consumer': 'purple',
        'Industrial': 'brown',
        'Utilities': 'gray',
        'Materials': 'gold'
    }
    return colors.get(sector, 'black')

In [13]:
def generate_rotation_signals(rs_ratio, rs_momentum):
    """
    Generate trading signals based on sector rotation
    """
    
    signals = []
    current_date = rs_ratio.index[-1]
    
    # Get current positions
    current_rs = rs_ratio.loc[current_date]
    current_mom = rs_momentum.loc[current_date]
    
    # Classify sectors into quadrants
    for sector in rs_ratio.columns:
        quadrant = classify_quadrant(current_rs[sector], current_mom[sector])
        
        # Check rotation direction (comparing to previous period)
        prev_rs = rs_ratio.loc[rs_ratio.index[-2], sector]
        prev_mom = rs_momentum.loc[rs_momentum.index[-2], sector]
        prev_quadrant = classify_quadrant(prev_rs, prev_mom)
        
        if quadrant != prev_quadrant:
            direction = f"{prev_quadrant} → {quadrant}"
        else:
            direction = "Stable"
        
        signals.append({
            'sector': sector,
            'date': current_date,
            'j_rs_ratio': current_rs[sector],
            'j_rs_momentum': current_mom[sector],
            'quadrant': quadrant,
            'rotation_direction': direction,
            'action': generate_action(quadrant, direction)
        })
    
    return pd.DataFrame(signals)

def classify_quadrant(rs_ratio, rs_momentum):
    """Classify sector position in RRG"""
    if rs_ratio >= 100:
        if rs_momentum >= 0:
            return "Leading"
        else:
            return "Weakening"
    else:
        if rs_momentum >= 0:
            return "Improving"
        else:
            return "Lagging"

def generate_action(quadrant, direction):
    """Generate trading action based on position and rotation"""
    actions = {
        'Leading': 'Hold/Add to position',
        'Weakening': 'Reduce position or hedge',
        'Lagging': 'Avoid or short',
        'Improving': 'Consider initiating position'
    }
    
    # Special cases for rotations
    if direction == "Improving → Leading":
        return "Strong Buy - Rotation into Leading quadrant"
    elif direction == "Leading → Weakening":
        return "Sell - Leaving Leading quadrant"
    elif direction == "Weakening → Lagging":
        return "Short - Deteriorating into Lagging"
    elif direction == "Lagging → Improving":
        return "Watch for buy signal - Bottoming"
    
    return actions.get(quadrant, 'Hold')

In [14]:
def backtest_rrg_strategy(rs_ratio, rs_momentum, sector_returns, rebalance_freq='weekly'):
    """
    Backtest a sector rotation strategy based on RRG signals
    """
    
    portfolio = {}
    returns = []
    
    for i, date in enumerate(rs_ratio.index):
        if i % (7 if rebalance_freq == 'weekly' else 30) == 0:  # Rebalance
            # Get current signals
            signals = generate_rotation_signals(
                rs_ratio.iloc[:i+1], 
                rs_momentum.iloc[:i+1]
            )
            
            # Select sectors in Leading and Improving quadrants
            selected = signals[signals['quadrant'].isin(['Leading', 'Improving'])]
            
            if len(selected) > 0:
                # Equal weight selected sectors
                weight = 1.0 / len(selected)
                portfolio = {row['sector']: weight for _, row in selected.iterrows()}
            else:
                portfolio = {}
        
        # Calculate portfolio return for this day
        if portfolio:
            daily_ret = sum(
                weight * sector_returns.loc[date, sector] 
                for sector, weight in portfolio.items() 
                if sector in sector_returns.columns
            )
            returns.append(daily_ret)
    
    return pd.Series(returns, index=rs_ratio.index[-len(returns):])

In [15]:
class JobRRGDashboard:
    def __init__(self, update_frequency='daily'):
        self.update_frequency = update_frequency
        self.historical_data = None
        self.current_signals = None
        
    def update(self):
        """Fetch latest data and update RRG"""
        # Get latest job postings
        new_data = get_sector_monthly_jobs()
        
        # Update calculations
        self.rs_ratio = calculate_rs_ratio(new_data)
        self.rs_momentum = calculate_rs_momentum(self.rs_ratio)
        
        # Generate signals
        self.current_signals = generate_rotation_signals(
            self.rs_ratio, 
            self.rs_momentum
        )
        
        # Create visualization
        self.rrg_plot = create_job_rrg(
            self.rs_ratio, 
            self.rs_momentum, 
            date=self.rs_ratio.index[-1]
        )
        
        # Calculate confidence metrics
        self.confidence_scores = self.calculate_confidence()
        
    def calculate_confidence(self):
        """Calculate confidence in signals based on multiple factors"""
        confidence = {}
        
        for _, signal in self.current_signals.iterrows():
            sector = signal['sector']
            
            # Factor 1: Distance from quadrant boundaries
            rs_dist = abs(signal['j_rs_ratio'] - 100)
            mom_dist = abs(signal['j_rs_momentum'])
            
            # # Factor 2: Job posting volume (more postings = more reliable)
            # volume_factor = min(self.get_sector_volume(sector) / 1000, 1.0)
            
            # # Factor 3: Breadth of hiring across firms in sector
            # breadth = self.get_sector_breadth(sector)
            
            # Composite confidence score
            confidence[sector] = (
                0.4 * (rs_dist / 30) +  # Normalize to 0-1
                0.3 * min(mom_dist / 2, 1)
            )
            
        return confidence

In [22]:
rrg = JobRRGDashboard()
rrg.update()

In [17]:
new_data = get_sector_monthly_jobs()

In [20]:
# Update calculations
rs_ratio = calculate_rs_ratio(new_data)
rs_momentum = calculate_rs_momentum(rs_ratio)

In [23]:
# Create visualization
rrg_plot = create_job_rrg(
    rs_ratio, 
    rs_momentum, 
    date=rs_ratio.index[-1]
)
rrg_plot